# Sport Recommender System ด้วย Graph
แนวคิดคือ ผู้ใช้ชอบหรือสนใจกีฬาชนิดไหน → หา User คนอื่นที่ชอบกีฬาคล้ายกัน → ดูว่าคนเหล่านั้นชอบกีฬาชนิดอะไรอีก → แนะนำกีฬาชนิดนั้นกลับมา

ตัวอย่างสถานการณ์ มีผู้ใช้ 4 คน

- Fai
- Phueng
- Pete
- Korn

และมีกีฬา

- Football
- Basketball
- Badminton
- Swimming
- Tennis

## 1. นำเข้าไลบรารี
ใช้ `networkx` สร้างกราฟ และ `matplotlib` วาดกราฟ

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

## 2. กำหนดข้อมูลว่าใครชอบกีฬาอะไร
เก็บข้อมูลเป็น dictionary โดย key คือชื่อผู้ใช้ และ value คือรายการกีฬาที่ชอบ

In [ ]:
users = ["Fai", "Phueng", "Pete", "Korn"]
sports = ["Football", "Basketball", "Badminton", "Swimming", "Tennis"]

likes = {
    "Fai":    ["Football", "Basketball"],
    "Phueng": ["Football", "Badminton", "Swimming"],
    "Pete":   ["Basketball", "Tennis"],
    "Korn":   ["Badminton", "Swimming", "Tennis"],
}

for user, fav in likes.items():
    print(f"{user} ชอบ: {fav}")

## 3. สร้างกราฟ
- Node คือผู้ใช้และกีฬา
- Edge คือความสัมพันธ์ "ผู้ใช้ชอบกีฬา"

In [ ]:
G = nx.Graph()
G.add_nodes_from(users, type="user")
G.add_nodes_from(sports, type="sport")

for user, fav_sports in likes.items():
    for sport in fav_sports:
        G.add_edge(user, sport)

print("จำนวน Node:", G.number_of_nodes())
print("จำนวน Edge:", G.number_of_edges())

## 4. วาดกราฟ
สีม่วงคือผู้ใช้ สีส้มคือกีฬา

In [ ]:
pos = nx.bipartite_layout(G, users)

plt.figure(figsize=(8, 6))
nx.draw_networkx_nodes(G, pos, nodelist=users, node_color="#8b6cff", node_size=1800)
nx.draw_networkx_nodes(G, pos, nodelist=sports, node_color="#ffb347", node_size=1800)
nx.draw_networkx_edges(G, pos, width=1.5, alpha=0.6)
nx.draw_networkx_labels(G, pos, font_size=10)
plt.title("User - Sport Graph")
plt.axis("off")
plt.show()

## 5. สร้างฟังก์ชันแนะนำกีฬา
ขั้นตอนการทำงาน
1. ดูว่าผู้ใช้เป้าหมายชอบกีฬาอะไร
2. หา User คนอื่นที่ชอบกีฬาซ้ำกัน
3. ดูว่าคนเหล่านั้นชอบกีฬาอะไรเพิ่มอีก
4. ให้คะแนนตามจำนวนกีฬาที่ชอบเหมือนกัน แล้วแนะนำกีฬาที่คะแนนสูงสุด

In [ ]:
def recommend_sport(target_user, top_n=3):
    my_sports = set(G.neighbors(target_user))
    scores = {}

    for other in users:
        if other == target_user:
            continue
        other_sports = set(G.neighbors(other))
        common = my_sports & other_sports   # กีฬาที่ชอบเหมือนกัน
        if not common:
            continue

        # กีฬาที่คนนั้นชอบ แต่ผู้ใช้เป้าหมายยังไม่มี
        for sport in other_sports - my_sports:
            scores[sport] = scores.get(sport, 0) + len(common)

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

## 6. ทดสอบระบบแนะนำกีฬา

In [ ]:
for user in users:
    result = recommend_sport(user)
    print(f"\n{user} ชอบ: {list(G.neighbors(user))}")
    if result:
        print("แนะนำ:")
        for sport, score in result:
            print(f"  - {sport} (คะแนน {score})")
    else:
        print("ยังไม่มีกีฬาที่แนะนำได้")

## 7. วาดกราฟพร้อมกีฬาที่แนะนำ
เลือกผู้ใช้ด้านล่าง แล้วดูเส้นประสีเขียวที่โยงไปยังกีฬาที่ระบบแนะนำ

In [ ]:
def show_recommendation(target_user):
    recs = [s for s, _ in recommend_sport(target_user)]
    my = list(G.neighbors(target_user))

    plt.figure(figsize=(8, 6))
    nx.draw_networkx_nodes(G, pos, nodelist=users, node_color="#8b6cff", node_size=1800)
    nx.draw_networkx_nodes(G, pos, nodelist=[s for s in sports if s not in recs],
                           node_color="#ffb347", node_size=1800)
    nx.draw_networkx_nodes(G, pos, nodelist=recs, node_color="#2dd4bf", node_size=1800)
    nx.draw_networkx_edges(G, pos, width=1, alpha=0.25)
    nx.draw_networkx_edges(G, pos, edgelist=[(target_user, s) for s in my],
                           width=3, edge_color="#8b6cff")
    nx.draw_networkx_edges(G, pos, edgelist=[(target_user, s) for s in recs],
                           width=3, edge_color="#2dd4bf", style="dashed")
    nx.draw_networkx_labels(G, pos, font_size=10)
    plt.title(f"แนะนำกีฬาให้ {target_user}  (เขียว = กีฬาที่แนะนำ)")
    plt.axis("off")
    plt.show()

show_recommendation("Fai")

## 8. ลองเปลี่ยนผู้ใช้เอง
แก้ชื่อในวงเล็บเป็น `"Phueng"`, `"Pete"` หรือ `"Korn"` แล้วกดรันอีกครั้ง

In [ ]:
user_name = "Korn"
print("แนะนำสำหรับ", user_name, ":", recommend_sport(user_name))
show_recommendation(user_name)